In [1]:
# Check GPU and mount Google Drive

from google.colab import drive
drive.mount("/content/drive")

import torch
from pathlib import Path

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001")

print("\nDataset:", ROOT)
print("Images:", len(list((ROOT / "images").glob("*.jpg"))))
print("transforms.json:", (ROOT / "transforms.json").exists())
print("rumahku PLY:", (ROOT / "rumahku_6000.ply").exists())

Mounted at /content/drive
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

Dataset: /content/drive/MyDrive/Omokai/digital_twin/room_001
Images: 152
transforms.json: True
rumahku PLY: True


In [3]:
# Install HLoc and SuperGlue

%cd /content

import os

if not os.path.exists("/content/Hierarchical-Localization"):
    !git clone -q https://github.com/cvg/Hierarchical-Localization.git

%cd /content/Hierarchical-Localization

!pip install -q -e .

if not os.path.exists(
    "/content/Hierarchical-Localization/third_party/SuperGluePretrainedNetwork"
):
    !git clone -q \
        https://github.com/magicleap/SuperGluePretrainedNetwork.git \
        third_party/SuperGluePretrainedNetwork

print("✓ HLoc ready")

/content
/content/Hierarchical-Localization
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 119.6 MB/s eta 0:00:00
✓ HLoc ready


In [4]:
# Extract SuperPoint features

from pathlib import Path
import sys

ROOT = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001")
WORK = Path("/content/digital_twin_3dgs")

WORK.mkdir(parents=True, exist_ok=True)

# Use the existing room images without duplicating them
IMAGE_DIR = ROOT / "images"

FEATURE_DIR = WORK / "hloc"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append("/content/Hierarchical-Localization")

from hloc import extract_features

feature_conf = extract_features.confs["superpoint_max"]

feature_path = extract_features.main(
    feature_conf,
    image_dir=IMAGE_DIR,
    export_dir=FEATURE_DIR,
    feature_path=FEATURE_DIR / "feats-superpoint-n4096-rmax1600.h5",
)

print("\n✓ SuperPoint extraction complete")
print("Features:", feature_path)

[2026/10/07 12:14:09 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2026/10/07 12:14:09 hloc INFO] Found 152 images in root /content/drive/MyDrive/Omokai/digital_twin/room_001/images.


ModuleNotFoundError: No module named 'SuperGluePretrainedNetwork.models'

In [5]:
from pathlib import Path

SG = Path("/content/Hierarchical-Localization/third_party/SuperGluePretrainedNetwork")

print("Repository exists:", SG.exists())
print("models directory:", (SG / "models").exists())

if SG.exists():
    print("\nRepository contents:")
    for p in sorted(SG.iterdir()):
        print(" ", p.name)

if (SG / "models").exists():
    print("\nModels contents:")
    for p in sorted((SG / "models").iterdir()):
        print(" ", p.name)

Repository exists: True
models directory: False

Repository contents:


In [7]:
# Reinstall the missing SuperGlue dependency

import shutil
from pathlib import Path

SG = Path("/content/Hierarchical-Localization/third_party/SuperGluePretrainedNetwork")

if SG.exists():
    shutil.rmtree(SG)

!git clone -q https://github.com/magicleap/SuperGluePretrainedNetwork.git \
    /content/Hierarchical-Localization/third_party/SuperGluePretrainedNetwork

print("✓ SuperGlue downloaded")
print("models exists:", (SG / "models").exists())
print("superpoint.py exists:", (SG / "models" / "superpoint.py").exists())

✓ SuperGlue downloaded
models exists: True
superpoint.py exists: True


In [8]:
# Extract SuperPoint features

from pathlib import Path
import sys

ROOT = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001")
WORK = Path("/content/digital_twin_3dgs")

IMAGE_DIR = ROOT / "images"
FEATURE_DIR = WORK / "hloc"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append("/content/Hierarchical-Localization")

from hloc import extract_features

feature_conf = extract_features.confs["superpoint_max"]

feature_path = extract_features.main(
    feature_conf,
    image_dir=IMAGE_DIR,
    export_dir=FEATURE_DIR,
    feature_path=FEATURE_DIR / "feats-superpoint-n4096-rmax1600.h5",
)

print("\n✓ SuperPoint extraction complete")
print("Features:", feature_path)

[2026/10/07 12:17:26 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2026/10/07 12:17:26 hloc INFO] Found 152 images in root /content/drive/MyDrive/Omokai/digital_twin/room_001/images.


Loaded SuperPoint model


100%|██████████| 152/152 [00:28<00:00,  5.27it/s]
[2026/10/07 12:17:55 hloc INFO] Finished exporting features.



✓ SuperPoint extraction complete
Features: /content/digital_twin_3dgs/hloc/feats-superpoint-n4096-rmax1600.h5


In [9]:
# Generate sequential image pairs

from pathlib import Path
import sys

WORK = Path("/content/digital_twin_3dgs")
IMAGE_DIR = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001/images")
PAIR_FILE = WORK / "hloc" / "pairs-sequential-10.txt"

sys.path.append("/content/Hierarchical-Localization")

from hloc import pairs_from_exhaustive

# HLoc's sequential pairing utility
from hloc import pairs_from_exhaustive

# Generate pairs with a temporal window of 10 images
pairs_from_exhaustive.main(
    output=PAIR_FILE,
    image_list=[p.name for p in sorted(IMAGE_DIR.glob("*.jpg"))],
)

print("✓ Pair file created:", PAIR_FILE)

[2026/10/07 12:18:39 hloc INFO] Found 11476 pairs.


✓ Pair file created: /content/digital_twin_3dgs/hloc/pairs-sequential-10.txt


In [11]:
# Create sequential pairs manually (10-frame overlap)

from pathlib import Path

WORK = Path("/content/digital_twin_3dgs")
IMAGE_DIR = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001/images")
PAIR_FILE = WORK / "hloc" / "pairs-sequential-10.txt"

images = sorted(p.name for p in IMAGE_DIR.glob("*.jpg"))

pairs = []

for i, img1 in enumerate(images):
    for j in range(i + 1, min(i + 11, len(images))):
        pairs.append(f"{img1} {images[j]}")

PAIR_FILE.write_text("\n".join(pairs))

print("✓ Sequential pair file created")
print("Images:", len(images))
print("Pairs:", len(pairs))
print("\nFirst 5 pairs:")
for pair in pairs[:5]:
    print(" ", pair)

✓ Sequential pair file created
Images: 152
Pairs: 1465

First 5 pairs:
  000000.jpg 000001.jpg
  000000.jpg 000002.jpg
  000000.jpg 000003.jpg
  000000.jpg 000004.jpg
  000000.jpg 000005.jpg


In [13]:
# Check the installed HLoc matching API

import inspect
from hloc import match_features

print(inspect.signature(match_features.main))

(conf: Dict, pairs: pathlib._local.Path, features: Union[pathlib._local.Path, str], export_dir: Optional[pathlib._local.Path] = None, matches: Optional[pathlib._local.Path] = None, features_ref: Optional[pathlib._local.Path] = None, overwrite: bool = False) -> pathlib._local.Path


In [14]:
# SuperGlue matching

from pathlib import Path
import sys

WORK = Path("/content/digital_twin_3dgs")
HL_DIR = WORK / "hloc"

sys.path.append("/content/Hierarchical-Localization")

from hloc import match_features

match_conf = match_features.confs["superglue"]

MATCH_PATH = HL_DIR / "matches-superglue.h5"

match_path = match_features.main(
    match_conf,
    pairs=HL_DIR / "pairs-sequential-10.txt",
    features=HL_DIR / "feats-superpoint-n4096-rmax1600.h5",
    export_dir=HL_DIR,
    matches=MATCH_PATH,
)

print("\n✓ SuperGlue matching complete")
print("Matches:", match_path)# CELL 5B — SuperGlue matching

from pathlib import Path
import sys

WORK = Path("/content/digital_twin_3dgs")
HL_DIR = WORK / "hloc"

sys.path.append("/content/Hierarchical-Localization")

from hloc import match_features

match_conf = match_features.confs["superglue"]

MATCH_PATH = HL_DIR / "matches-superglue.h5"

match_path = match_features.main(
    match_conf,
    pairs=HL_DIR / "pairs-sequential-10.txt",
    features=HL_DIR / "feats-superpoint-n4096-rmax1600.h5",
    export_dir=HL_DIR,
    matches=MATCH_PATH,
)

print("\n✓ SuperGlue matching complete")
print("Matches:", match_path)

[2026/10/07 12:23:19 hloc INFO] Matching local features with configuration:
{'model': {'name': 'superglue',
           'sinkhorn_iterations': 50,
           'weights': 'outdoor'},
 'output': 'matches-superglue'}


Loaded SuperGlue model ("outdoor" weights)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 5 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  0%|          | 0/1465 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 5 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
10


✓ SuperGlue matching complete
Matches: /content/digital_twin_3dgs/hloc/matches-superglue.h5

✓ SuperGlue matching complete
Matches: /content/digital_twin_3dgs/hloc/matches-superglue.h5


In [16]:
# Inspect the installed HLoc reconstruction API

import inspect
from hloc import reconstruction

print(inspect.signature(reconstruction.main))

(sfm_dir: pathlib._local.Path, image_dir: pathlib._local.Path, pairs: pathlib._local.Path, features: pathlib._local.Path, matches: pathlib._local.Path, camera_mode: pycolmap._core.CameraMode = CameraMode.AUTO, verbose: bool = False, skip_geometric_verification: bool = False, min_match_score: Optional[float] = None, image_list: Optional[List[str]] = None, image_options: Optional[Dict[str, Any]] = None, mapper_options: Optional[Dict[str, Any]] = None) -> pycolmap._core.Reconstruction


In [17]:
# CELL 6B — Build and SAVE the SfM reconstruction

from pathlib import Path
import sys

ROOT = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001")
WORK = Path("/content/digital_twin_3dgs")
HL_DIR = WORK / "hloc"

# Save the reconstruction directly to Google Drive
SFM_OUTPUT = ROOT / "sfm"
SFM_OUTPUT.mkdir(parents=True, exist_ok=True)

sys.path.append("/content/Hierarchical-Localization")

from hloc import reconstruction

sfm_model = reconstruction.main(
    sfm_dir=SFM_OUTPUT,
    image_dir=ROOT / "images",
    pairs=HL_DIR / "pairs-sequential-10.txt",
    features=HL_DIR / "feats-superpoint-n4096-rmax1600.h5",
    matches=HL_DIR / "matches-superglue.h5",
)

print("\n✓ SfM reconstruction complete")
print("Saved to:", SFM_OUTPUT)

[2026/10/07 12:33:32 hloc INFO] Writing COLMAP logs to /content/drive/MyDrive/Omokai/digital_twin/room_001/sfm/colmap.LOG.*
[2026/10/07 12:33:32 hloc INFO] Creating an empty database...
[2026/10/07 12:33:32 hloc INFO] Importing images into the database...
[2026/10/07 12:33:38 hloc INFO] Importing features into the database...
100%|██████████| 152/152 [00:00<00:00, 484.43it/s]
[2026/10/07 12:33:38 hloc INFO] Importing matches into the database...
100%|██████████| 1465/1465 [00:02<00:00, 494.58it/s]
[2026/10/07 12:33:41 hloc INFO] Performing geometric verification of the matches...
[2026/10/07 12:33:54 hloc INFO] Running 3D reconstruction...
Reconstruction 3:   6%|▌         | 9/152 [00:13<03:39,  1.54s/images, registered]
[2026/10/07 12:46:31 hloc INFO] Reconstructed 1 model(s).
[2026/10/07 12:46:31 hloc INFO] Largest model is #0 with 142 images.
[2026/10/07 12:46:31 hloc INFO] Reconstruction statistics:
Reconstruction:
	num_rigs = 142
	num_cameras = 142
	num_frames = 142
	num_reg_frames


✓ SfM reconstruction complete
Saved to: /content/drive/MyDrive/Omokai/digital_twin/room_001/sfm


In [19]:
from pathlib import Path
import pycolmap

SFM_DIR = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001/sfm")

print("Loading COLMAP model from:")
print(SFM_DIR)

rec = pycolmap.Reconstruction(SFM_DIR)

print("\n=== SfM MODEL ===")
print("Registered images:", len(rec.images))
print("Cameras:", len(rec.cameras))
print("3D points:", len(rec.points3D))

errors = [p.error for p in rec.points3D.values() if p.error is not None]

if errors:
    print("Mean reprojection error:", sum(errors) / len(errors))

print("\n✓ COLMAP model loaded successfully")

Loading COLMAP model from:
/content/drive/MyDrive/Omokai/digital_twin/room_001/sfm

=== SfM MODEL ===
Registered images: 142
Cameras: 142
3D points: 29666
Mean reprojection error: 1.2341412101693618

✓ COLMAP model loaded successfully


In [21]:
from pathlib import Path
import json
import numpy as np
import pycolmap

ROOT = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001")
SFM_DIR = ROOT / "sfm"
NS_DIR = ROOT / "nerfstudio_data"

NS_DIR.mkdir(parents=True, exist_ok=True)

rec = pycolmap.Reconstruction(SFM_DIR)

print("Registered images:", len(rec.images))
print("3D points:", len(rec.points3D))

# Use the first COLMAP camera
cam = next(iter(rec.cameras.values()))

print("\nCamera model:", cam.model.name)
print("Resolution:", cam.width, "x", cam.height)
print("Params:", cam.params)

frames = []

for image_id, image in sorted(rec.images.items()):
    camera = rec.cameras[image.camera_id]

    # In this pycolmap version cam_from_world is callable
    cam_from_world = image.cam_from_world()

    # COLMAP world -> camera
    R = cam_from_world.rotation.matrix()
    t = np.asarray(cam_from_world.translation)

    # Convert camera -> world
    R_c2w = R.T
    t_c2w = -R_c2w @ t

    transform = np.eye(4)
    transform[:3, :3] = R_c2w
    transform[:3, 3] = t_c2w

    frames.append({
        "file_path": f"images/{image.name}",
        "transform_matrix": transform.tolist()
    })

transforms = {
    "camera_model": cam.model.name,
    "fl_x": float(cam.params[0]),
    "fl_y": float(cam.params[0]),
    "cx": float(cam.params[1]),
    "cy": float(cam.params[2]),
    "w": int(cam.width),
    "h": int(cam.height),
    "frames": frames
}

OUT = NS_DIR / "transforms.json"

with open(OUT, "w") as f:
    json.dump(transforms, f, indent=2)

print("\n✓ transforms.json created")
print("Path:", OUT)
print("Frames:", len(frames))

Registered images: 142
3D points: 29666

Camera model: SIMPLE_RADIAL
Resolution: 1920 x 1080
Params: [ 1.41001151e+03  9.60000000e+02  5.40000000e+02 -4.21772611e-03]

✓ transforms.json created
Path: /content/drive/MyDrive/Omokai/digital_twin/room_001/nerfstudio_data/transforms.json
Frames: 142


In [22]:
from pathlib import Path
import json

NS_DIR = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001/nerfstudio_data")
IMAGE_DIR = NS_DIR / "images"

# Create image directory
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Link/copy the registered images from the original dataset
SOURCE_DIR = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001/images")

with open(NS_DIR / "transforms.json") as f:
    data = json.load(f)

frame_names = [
    Path(frame["file_path"]).name
    for frame in data["frames"]
]

source_names = {p.name for p in SOURCE_DIR.glob("*.jpg")}

missing = [name for name in frame_names if name not in source_names]

print("Frames in transforms:", len(frame_names))
print("Source images:", len(source_names))
print("Missing:", len(missing))

if missing:
    print("\nMissing images:")
    for name in missing[:20]:
        print(" ", name)
else:
    print("\n✓ All 142 registered images are available")

Frames in transforms: 142
Source images: 152
Missing: 0

✓ All 142 registered images are available


In [24]:
from pathlib import Path
import json
import shutil

ROOT = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001")
NS_DIR = ROOT / "nerfstudio_data"
NS_IMG_DIR = NS_DIR / "images"
SOURCE_DIR = ROOT / "images"

NS_IMG_DIR.mkdir(parents=True, exist_ok=True)

with open(NS_DIR / "transforms.json") as f:
    data = json.load(f)

registered = {
    Path(frame["file_path"]).name
    for frame in data["frames"]
}

# Clean existing dataset images
for p in NS_IMG_DIR.iterdir():
    if p.is_file():
        p.unlink()

# Copy only registered images
for name in sorted(registered):
    src = SOURCE_DIR / name
    dst = NS_IMG_DIR / name

    if not src.exists():
        print("Missing:", name)
        continue

    shutil.copy2(src, dst)

images = list(NS_IMG_DIR.glob("*.jpg"))

print("Registered images:", len(registered))
print("Copied images:", len(images))

missing = [
    name for name in registered
    if not (NS_IMG_DIR / name).exists()
]

print("Missing:", len(missing))

if not missing:
    print("\n✓ Nerfstudio dataset prepared successfully")

Registered images: 142
Copied images: 142
Missing: 0

✓ Nerfstudio dataset prepared successfully


In [25]:
from pathlib import Path
import json
from PIL import Image

NS_DIR = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001/nerfstudio_data")

with open(NS_DIR / "transforms.json") as f:
    data = json.load(f)

frames = data["frames"]

print("=== Nerfstudio Dataset Check ===")
print("Frames:", len(frames))
print("Images:", len(list((NS_DIR / "images").glob("*.jpg"))))
print("Resolution:", data["w"], "x", data["h"])
print("Camera model:", data["camera_model"])
print("Focal length:", data["fl_x"])
print("Principal point:", data["cx"], data["cy"])

# Check a few actual images
bad = []

for frame in frames:
    path = NS_DIR / frame["file_path"]
    try:
        with Image.open(path) as im:
            if im.size != (data["w"], data["h"]):
                bad.append((path.name, im.size))
    except Exception as e:
        bad.append((path.name, str(e)))

print("Bad images:", len(bad))

if bad:
    print("\nFirst problems:")
    for item in bad[:10]:
        print(item)
else:
    print("\n✓ All images and camera metadata are consistent")

=== Nerfstudio Dataset Check ===
Frames: 142
Images: 142
Resolution: 1920 x 1080
Camera model: SIMPLE_RADIAL
Focal length: 1410.011514711749
Principal point: 960.0 540.0
Bad images: 0

✓ All images and camera metadata are consistent


In [26]:
import sys
import subprocess

try:
    import nerfstudio
    print("Nerfstudio:", nerfstudio.__version__)
except Exception:
    print("Nerfstudio is not installed.")
    print("Installing...")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "nerfstudio"],
        check=True
    )

    import nerfstudio
    print("Nerfstudio:", nerfstudio.__version__)

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

Nerfstudio is not installed.
Installing...


KeyboardInterrupt: 

In [27]:
pip install nerfstudio

  Using cached nerfstudio-1.1.5-py3-none-any.whl.metadata (24 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached av-19.0.1-cp312-abi3-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached comet_ml-3.58.7-py3-none-any.whl.metadata (4.0 kB)
  Using cached tyro-1.0.16-py3-none-any.whl.metadata (12 kB)
  Using cached ninja-1.13.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.4 kB)
  Using cached jaxtyping-0.3.11-py3-none-any.whl.metadata (6.2 kB)
  Using cached jupyterlab-4.6.4-py3-none-any.whl.metadata (16 kB)
  Using cached mediapy-1.2.7-py3-none-any.whl.metadata (4.8 kB)
  Using cached msgpack_numpy-0.4.8-py2.py3-none-any.whl.metadata (5.0 kB)
  Using cached nerfacc-0.5.2-py3-none-any.whl.metadata (877 bytes)
  Using cached open3d-0.20.0-cp313-cp313-manylinux_2_35_x86_64.whl.metadata (4.7 kB)
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cach

In [28]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [29]:
!pip install -q --no-deps nerfstudio==1.1.5 gsplat==1.4.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.8/580.8 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 20.5 MB/s eta 0:00:00


In [31]:
import torch
import nerfstudio
import gsplat

print("Nerfstudio import: OK")
print("gsplat import: OK")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Nerfstudio import: OK
gsplat import: OK
PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4


In [32]:
from pathlib import Path
import json

DATASET = Path("/content/drive/MyDrive/Omokai/digital_twin/room_001/nerfstudio_data")

with open(DATASET / "transforms.json") as f:
    meta = json.load(f)

images = DATASET / "images"

print("Dataset:", DATASET)
print("Images:", len(list(images.glob("*.jpg"))))
print("Frames in transforms:", len(meta["frames"]))
print("Resolution:", meta["w"], "x", meta["h"])
print("Camera model:", meta["camera_model"])
print("Focal length:", meta["fl_x"], meta["fl_y"])
print("Dataset ready:", (images.exists() and (DATASET / "transforms.json").exists()))

Dataset: /content/drive/MyDrive/Omokai/digital_twin/room_001/nerfstudio_data
Images: 142
Frames in transforms: 142
Resolution: 1920 x 1080
Camera model: SIMPLE_RADIAL
Focal length: 1410.011514711749 1410.011514711749
Dataset ready: True


In [34]:
!pip install -q tyro

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nerfstudio 1.1.5 requires appdirs>=1.4, which is not installed.
nerfstudio 1.1.5 requires av>=9.2.0, which is not installed.
nerfstudio 1.1.5 requires comet-ml>=3.33.8, which is not installed.
nerfstudio 1.1.5 requires fpsample, which is not installed.
nerfstudio 1.1.5 requires jaxtyping>=0.2.15, which is not installed.
nerfstudio 1.1.5 requires jupyterlab>=3.3.4, which is not installed.
nerfstudio 1.1.5 requires mediapy>=1.1.0, which is not installed.
nerfstudio 1.1.5 requires msgpack-numpy>=0.4.8, which is not installed.
nerfstudio 1.1.5 requires nerfacc==0.5.2, which is not installed.
nerfstudio 1.1.5 requires ninja>=1.10, which is not installed.
nerfstudio 1.1.5 requires nuscenes-devkit>=1.1.1, which is not installed.
nerfstudio 1

In [33]:
!ns-train splatfacto \
    --data /content/drive/MyDrive/Omokai/digital_twin/room_001/nerfstudio_data \
    --output-dir /content/digital_twin_3dgs/runs \
    --max-num-iterations 1000 \
    nerfstudio-data

Traceback (most recent call last):
  File "/usr/local/bin/ns-train", line 5, in <module>
    from nerfstudio.scripts.train import entrypoint
  File "/usr/local/lib/python3.13/dist-packages/nerfstudio/scripts/train.py", line 58, in <module>
    import tyro
ModuleNotFoundError: No module named 'tyro'
